In [ ]:
import sys, os
import numpy as np
import pandas as pd
from scipy import stats

from csc import *
from exp_utils import *

current_dir = os.path.dirname(os.path.abspath('__file__'))
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import tol_colors as tc

In [2]:
import scienceplots
plt.style.use(['science', 'no-latex'])
plt.rcParams['text.latex.preamble'] = r'\usepackage[cm]{sfmath}\usepackage{amsmath}\centering'
plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.it'] = 'Helvetica:italic'
plt.rcParams["text.usetex"] = False


In [3]:
potato_duplicate_questions = [14, 83, 121]
models = [
    "gemma-2-9b-it",
    "gemma-3-12b-it",
    "Llama-3.1-8B-Instruct",
    "Mistral-7B-Instruct-v0.3",
    "Phi-3.5-mini-instruct",
]

model_rename = {
    "gemma-2-9b-it": "Gemma-2-9B",
    "gemma-3-12b-it": "Gemma-3-12B",
    "Llama-3.1-8B-Instruct": "Llama-3.1-8B",
    "Mistral-7B-Instruct-v0.3": "Mistral-7B",
    "Phi-3.5-mini-instruct": "Phi-3.5-3.8B",
}

coverage_methods = {
    "num_sets": "NumSets",
    "gt": "Good-Turing",
    "ueigv": "$U_{EigV}$",
    "hybrid": "Hybrid"
}

datasets = {
    "hotpot_qa_final": "HotpotQA",
    "squad_v2_final": "SQuAD 2.0",
    "potato_final": "POTATO",
    "bioasq_final": "BioASQ",
}
pm_symbol = u"\u00B1"
num_samples_list = [5, 10, 25, 50, 75, 100]

#### Ratio of alphabet size estimator to A*

In [4]:
p = "no_preprompt"

In [ ]:
square = False

if square:
    fig = plt.figure(figsize=(10, 10))
    gs = gridspec.GridSpec(3, 3, figure=fig)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[0, 1])
    ax4 = fig.add_subplot(gs[1, 0])
    ax5 = fig.add_subplot(gs[1, 1])
else:
    fig = plt.figure(figsize=(25, 5))
    gs = gridspec.GridSpec(1, 5, figure=fig)
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[0, 1])
    ax3 = fig.add_subplot(gs[0, 2])
    ax4 = fig.add_subplot(gs[0, 3])
    ax5 = fig.add_subplot(gs[0, 4])
    
axes = [ax1, ax2, ax3, ax4, ax5]

colors = tc.get_colorset('muted')
dataset_colors = {
    "HotpotQA": colors[0],
    "SQuAD 2.0": colors[1],
    "POTATO": colors[3],
    "BioASQ": colors[4]
}

large_fontsize = 35
medium_fontsize = 30
small_fontsize = 25

for model_idx in range(len(models)):
    model = models[model_idx]
    ax = axes[model_idx]
    ax.axhline(
        y=1., 
        linestyle=':', 
        color='grey', 
        label='$\\langle\widehat{|S|}/S^*\\rangle=1$'
    )

    alphabet_df = pd.read_csv(f"{current_dir}/data/{p}/{model}/alphabet.csv")

    aggregate_means_hybrid = None
    aggregate_means_plugin = None

    for dataset in datasets:
        means_hybrid = []
        means_plugin = []

        for n in num_samples_list:
            df_subset = alphabet_df[(alphabet_df["dataset"]==dataset)&(alphabet_df["n"]==n)&(alphabet_df["oracle"]!=0)]
            ratios_hybrid = df_subset["hybrid"]/df_subset["oracle"]
            ratios_plugin = df_subset["num_sets"]/df_subset["oracle"]
            means_hybrid.append(ratios_hybrid.mean())
            means_plugin.append(ratios_plugin.mean())
        
        if aggregate_means_hybrid is None:
            aggregate_means_hybrid = np.array(means_hybrid)
        else:
            aggregate_means_hybrid += np.array(means_hybrid)

        if aggregate_means_plugin is None:
            aggregate_means_plugin = np.array(means_plugin)
        else:
            aggregate_means_plugin += np.array(means_plugin)

    aggregate_means_hybrid /= len(datasets)
    aggregate_means_plugin /= len(datasets)

    ax.plot(
        num_samples_list, aggregate_means_hybrid, 
        "o", markersize=12,
        label="$\\widehat{|S|}_{Hybrid}$ (Ours)",
        color=colors.indigo,
        linestyle=None,
        lw=4
    )

    ax.plot(
        num_samples_list, aggregate_means_plugin, 
        "x", markersize=12,
        label="$NumSets$",
        color=colors.indigo,
        linestyle=":",
        lw=4
    )

    sns.despine(top=True, right=True, left=False, bottom=False, ax=ax)
    ax.xaxis.set_minor_locator(plt.NullLocator())
    ax.yaxis.set_minor_locator(plt.NullLocator())
    ax.tick_params(axis="y", which="both", right=False)
    ax.tick_params(axis="x", which="both", top=False) 

    ax.set_title(model_rename[model], fontsize=large_fontsize)
    ax.set_xlabel("$n$", fontsize=large_fontsize)
    if model_idx == 0:
        ax.set_ylabel("$\\langle\widehat{|S|}/S^*\\rangle$", fontsize=large_fontsize)
    ax.set_ylim(0.2, 1.19)
    ax.set_xscale("log")
    ax.set_xticks([5, 10, 25, 50, 100])
    ax.set_xticklabels([5, 10, 25, 50, 100])
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0, 1.2])
    ax.tick_params(axis='both', which='both', labelsize=small_fontsize)

handles, labels = ax.get_legend_handles_labels()
handles = [handles[1], handles[2], handles[0]]
labels = [labels[1], labels[2], labels[0]]
# interleave for 2 column setup
if square:
    handles = [handles[i] for i in range(1, len(handles), 2)]+[handles[i] for i in range(0, len(handles), 2)]
    labels = [labels[i] for i in range(1, len(labels), 2)]+[labels[i] for i in range(0, len(labels), 2)]
fig.legend(
    handles, labels, fontsize=medium_fontsize, loc='lower center', 
        bbox_to_anchor=(0.5, -0.25), ncol=3
)

plt.tight_layout()
plt.savefig('figures/alphabet_ratios_plugin_summary_draft.pdf')